# Memory and state

Assistant version 7: Keep conversation state and customer preferences across sessions.

Read `README.md` for the analogies, diagrams, API explanations, common mistakes and quiz. These cells contain the same source as the scripts; edit inputs and rerun a cell to explore. Outputs are intentionally cleared in Git. This notebook executes scripted fixtures with network blocked, not real models. Use the script commands without `--offline` for live experiments.

Start with `uv sync --extra agents --group authoring`, then `uv run --extra agents --group authoring jupyter lab` from the repository root.

In [ ]:
import asyncio
import os
import sys
from pathlib import Path

root = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
os.chdir(root)
sys.path.insert(0, str(root))
os.environ["COURSE_OFFLINE"] = "true"
os.environ["LANGSMITH_TRACING"] = "false"
from common.testing import offline_environment

print("OFFLINE FIXTURES: no API fees; outputs do not measure model quality.")

## Stateless models; InMemorySaver; thread_id and conversation isolation

Predict the output before running. The function definitions below match `examples/01_stateless_threads.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Contrast stateless calls with isolated checkpoint threads.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/01_stateless_threads.py --offline
Expected: Same-thread history grows; another thread starts fresh.
Cost: Six short calls; offline no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import thread_config
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    for saver in [None, InMemorySaver()]:
        agent = create_agent(get_chat_model(settings=settings), tools=[], checkpointer=saver)
        for thread, text in [
            ("first", "My preferred shipping is standard."),
            ("first", "What did I say?"),
            ("other", "What did I say?"),
        ]:
            # KEY IDEA: the host owns customer/thread scope; a thread ID is not authentication.
            result = agent.invoke(
                {"messages": [{"role": "user", "content": text}]}, thread_config("C001", thread)
            )
            print_text(
                f"saved={saver is not None}, thread={thread}, messages={len(result['messages'])}"
            )
    print_text("Offline replies are fixed; inspect retained messages, not recall quality.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## SqliteSaver lifecycle; resume in a fresh process

Predict the output before running. The function definitions below match `examples/02_sqlite_checkpoint.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Reopen a durable checkpoint in a fresh Python process.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/02_sqlite_checkpoint.py --offline
Expected: A child process sees persisted messages without making a model call.
Cost: One generation; subprocess reads local state only.
Requires: agents
"""

import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

from langchain.agents import create_agent
from langgraph.checkpoint.sqlite import SqliteSaver

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import thread_config
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    with TemporaryDirectory() as folder:
        path = Path(folder) / "threads.db"
        config = thread_config("C001", "demo")
        with SqliteSaver.from_conn_string(str(path)) as saver:
            agent = create_agent(get_chat_model(settings=settings), tools=[], checkpointer=saver)
            agent.invoke(
                {"messages": [{"role": "user", "content": "Remember standard shipping."}]}, config
            )
        # KEY IDEA: a closed/reopened connection, even in a new process, retains state.
        code = """
from langgraph.checkpoint.sqlite import SqliteSaver
import sys
with SqliteSaver.from_conn_string(sys.argv[1]) as saver:
    checkpoint = saver.get({'configurable': {'thread_id': 'C001:demo'}})
    print(len(checkpoint['channel_values']['messages']))
"""
        result = subprocess.run(
            [sys.executable, "-c", code, str(path)],
            capture_output=True,
            text=True,
            timeout=20,
            check=True,
        )
        print_text("Fresh process retained messages: " + result.stdout.strip())


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## trim_messages; token budget; preserve matching tool-call/result pairs

Predict the output before running. The function definitions below match `examples/03_trim.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Trim old turns while preserving tool-call/result pairs.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/03_trim.py --offline
Expected: Latest complete turn remains within an approximate token budget.
Cost: No model calls. Approximate token counts are not provider billing.
Requires: agents
"""

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage

from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.memory import trim_history
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    messages = [
        SystemMessage("Use only Brightwave evidence."),
        HumanMessage("Old discussion. " * 100),
        AIMessage("Earlier reply."),
        HumanMessage("Check BW-001 stock."),
        tool_request("check_stock", {"sku": "BW-001"}),
        ToolMessage(content="87", tool_call_id="call-1"),
    ]
    # KEY IDEA: discard complete older turns rather than orphaning tool messages.
    trimmed = trim_history(messages, 160)
    print_text(str([(m.type, m.content) for m in trimmed]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## SummarizationMiddleware; summarize older history without losing current task

Predict the output before running. The function definitions below match `examples/04_summarize.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Summarize older history while retaining the current task.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/04_summarize.py --offline
Expected: A summary marker replaces older turns; the latest user request survives.
Cost: One summary call plus one answer call; offline no API fee.
Requires: agents
"""

from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    summary = get_chat_model(
        settings=settings, responses=["Customer prefers standard shipping; now asks about returns."]
    )
    # KEY IDEA: summarization is another model call and can lose details; inspect the result.
    agent = create_agent(
        get_chat_model(settings=settings),
        tools=[],
        middleware=[
            SummarizationMiddleware(summary, trigger=("messages", 4), keep=("messages", 2))
        ],
    )
    messages = [
        {"role": "user", "content": "I prefer standard shipping."},
        {"role": "assistant", "content": "Recorded."},
        {"role": "user", "content": "I bought a tent."},
        {"role": "assistant", "content": "How can I help?"},
        {"role": "user", "content": "What is the return window?"},
    ]
    result = agent.invoke({"messages": messages})
    for message in result["messages"]:
        print_text(str((message.type, message.content)))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Custom AgentState; separate state from runtime context

Predict the output before running. The function definitions below match `examples/05_custom_state.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Keep evolving task state separate from immutable customer context.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/05_custom_state.py --offline
Expected: The result retains a typed task note; identity stays in runtime context.
Cost: Up to five short model calls; offline fixtures have no API fee.
Requires: agents
"""

from pathlib import Path
from typing import NotRequired

from langchain.agents import AgentState, create_agent

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_chat_model
from common.pretty import print_text
from common.returns import SupportContext


class SupportState(AgentState):
    task_note: NotRequired[str]


def main() -> None:
    settings = lesson_settings()
    # KEY IDEA: mutable task notes belong in state; authenticated identity belongs in context.
    agent = create_agent(
        get_chat_model(settings=settings),
        tools=[],
        state_schema=SupportState,
        context_schema=SupportContext,
    )
    result = agent.invoke(
        {
            "messages": [{"role": "user", "content": "Help with my return."}],
            "task_note": "Awaiting SKU",
        },
        context=SupportContext("C001", Path("unused")),
    )
    print_text(str({"task_note": result["task_note"], "messages": len(result["messages"])}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Store namespaces; InMemoryStore; semantic search of preferences across threads

Predict the output before running. The function definitions below match `examples/06_long_term_store.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Scope a semantic preference store by authenticated customer.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/06_long_term_store.py --offline
Expected: Only C001 namespace results appear; fake-vector scores are not semantic evidence.
Cost: Embedding calls only; offline no API fee.
Requires: agents
"""

from langgraph.store.memory import InMemoryStore

from common.errors import run_cli
from common.lesson import lesson_settings
from common.models import get_embeddings
from common.pretty import print_text


def main() -> None:
    settings = lesson_settings()
    embeddings = get_embeddings(settings=settings)
    # KEY IDEA: namespace filtering is authorization scope; vector ranking is only relevance.
    store = InMemoryStore(
        index={
            "dims": len(embeddings.embed_query("dimension probe")),
            "embed": embeddings,
            "fields": ["text"],
        }
    )
    store.put(("customers", "C001"), "shipping", {"text": "Prefers standard shipping"})
    store.put(("customers", "C002"), "shipping", {"text": "Prefers expedited shipping"})
    results = store.search(("customers", "C001"), query="delivery preference", limit=2)
    print_text(str([(r.namespace, r.value, r.score) for r in results]))
    print_text("InMemoryStore does not survive a process restart. Fake ranking is non-semantic.")


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## SQLite-backed preference tool for no-service persistence; preview PostgresStore in module 20

Predict the output before running. The function definitions below match `examples/07_durable_preferences.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Persist a scoped preference with SQLite and reopen it.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/examples/07_durable_preferences.py --offline
Expected: C001 shipping survives reopening; C002 has no preference.
Cost: No model calls or API fees.
Requires: agents
"""

from pathlib import Path
from tempfile import TemporaryDirectory

from common.errors import run_cli
from common.lesson import lesson_settings
from common.memory import Preferences
from common.pretty import print_text


def main() -> None:
    lesson_settings()
    with TemporaryDirectory() as folder:
        path = Path(folder) / "preferences.db"
        Preferences(path).put("C001", "standard")
        # KEY IDEA: this is an application repository, not a LangGraph store implementation.
        reopened = Preferences(path)
        print_text(str({"C001": reopened.get("C001"), "C002": reopened.get("C002")}))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Real-world use case: Keep conversation state and customer preferences across sessions

Predict the output before running. The function definitions below match `use_case/remembering_agent.py`. The notebook runs `main` in a worker thread because script examples that use `asyncio.run` need their own event loop. This wrapper does not change the taught mechanism.

In [ ]:
"""Remember customer preferences across separate conversation threads.

Run from the repository root (join wrapped command lines):
uv run --extra agents python modules/11_memory/use_case/remembering_agent.py --offline
Expected: A second thread retrieves the stored preference using trusted customer context.
Cost: Four short model calls; offline no API fee.
Requires: agents
"""

from pathlib import Path

from langchain.agents import create_agent
from langchain.tools import ToolRuntime, tool
from langgraph.checkpoint.sqlite import SqliteSaver

from common.config import ROOT
from common.errors import run_cli
from common.fixtures import tool_request
from common.lesson import lesson_settings
from common.memory import Preferences, thread_config
from common.models import get_chat_model
from common.pretty import print_text
from common.returns import SupportContext


def main() -> None:
    settings = lesson_settings()
    folder = ROOT / settings.runtime_dir / "remembering-agent"
    folder.mkdir(parents=True, exist_ok=True)
    preferences = Preferences(Path(folder) / "preferences.db")
    if preferences.get("C001") is None:
        preferences.put("C001", "standard")

    @tool
    def read_preference(runtime: ToolRuntime[SupportContext]) -> dict:
        """Read only the authenticated customer's stored shipping preference."""
        return {"shipping": preferences.get(runtime.context.customer_id)}

    with SqliteSaver.from_conn_string(str(Path(folder) / "threads.db")) as saver:
        for thread in ["morning", "evening"]:
            # KEY IDEA: independent conversation history can share customer-scoped preferences.
            agent = create_agent(
                get_chat_model(
                    settings=settings,
                    responses=[
                        tool_request("read_preference", {}),
                        "Use the preference returned by the tool.",
                    ],
                ),
                [read_preference],
                context_schema=SupportContext,
                checkpointer=saver,
            )
            result = agent.invoke(
                {"messages": [{"role": "user", "content": "What shipping do I prefer?"}]},
                thread_config("C001", thread),
                context=SupportContext("C001", Path("unused")),
            )
            print_text(str([m.content for m in result["messages"] if m.type == "tool"]))


sys.argv = ["notebook", "--offline"]
with offline_environment():
    await asyncio.to_thread(main)

## Practice and reflect

Change a Brightwave input and rerun the relevant cell. Inspect whether the prompt, tool arguments or parser result changed. A scripted model reply may stay the same: that is a fixture limitation, not evidence that a real model ignores input. Complete the three exercises in `exercises/`, run their target checks, and answer the quiz in `quiz.md`. Leave outputs cleared before committing.

Run `uv run --extra agents pytest modules/11_memory/tests` for the supplied solution checks.